# 🚀 TASK 6.6: FINAL LOCKED BENCHMARK & POST-HOC COMPARATIVE EVALUATION
### BÀI KIỂM CHUẨN KHÓA SỔ CHUNG CUỘC TRÊN 18 TẬP TEST & ĐỐI CHIẾU 3 BÊN MINH BẠCH

> **Mục tiêu**: Nạp checkpoint rút gọn `fatformer_crossdomain_weights_only.pth` (2.31 GB) cùng cặp tham số đóng băng $(T=1.0811, T^*=0.3700)$ từ Task 6.4 để đo đạc đối chiếu hậu nghiệm 2 chiều trên 18 tập test benchmark Clean và 6 biến thể Degraded, tính toán khoảng tin cậy 95% Bootstrap.
> **Môi trường**: Kaggle Server GPU / Colab (Internet: Off, 100% Offline).
> **Cơ chế tải kết quả**: Sau khi chạy xong (hoặc bấm **Save Version**), toàn bộ kết quả nằm trong tab **Output** của notebook trên Kaggle. Bạn có thể tải tệp `task_6.6_final_results.zip` hoặc xem các file CSV/PNG trực tiếp trên giao diện Kaggle.

## 🛠️ BƯỚC 1: THIẾT LẬP MÃ NGUỒN & TỰ ĐỘNG BUNG CÔNG CỤ ĐÁNH GIÁ (OFFLINE 100%)
> Giải nén mã nguồn nhẹ, tự động bung công cụ đánh giá `tools/eval_locked_benchmark.py` và `tools/plot_three_way_benchmark.py` mà không phụ thuộc vào gói code cũ.

In [ ]:
# ==============================================================================
# BƯỚC 1: THIẾT LẬP MÃ NGUỒN VÀ BUNG CÁC CÔNG CỤ ĐÁNH GIÁ (OFFLINE 100%)
# ==============================================================================
import os
import sys
import glob
import json
import shutil
import zipfile
import base64
import zlib

print("=" * 80)
print("BƯỚC 1: ĐỒNG BỘ MÃ NGUỒN FATFORMER-XLA TRÊN KAGGLE (OFFLINE 100%)")
print("=" * 80)

WORK_DIR = "/kaggle/working/fatformer-xla"
if os.path.exists(WORK_DIR):
    shutil.rmtree(WORK_DIR)
os.makedirs(WORK_DIR, exist_ok=True)

# Bộ lọc loại bỏ file nặng tránh tràn ổ đĩa
def ignore_heavy(dir_path, filenames):
    ignored = []
    for f in filenames:
        if f.endswith(('.pth', '.pt', '.ckpt', '.bin', '.tar', '.zip')):
            ignored.append(f)
        if f in ['checkpoints', 'pretrained', 'test_degraded']:
            ignored.append(f)
    return ignored

# 1. Tìm kiếm và sao chép mã nguồn nhẹ từ Kaggle Input
zip_cands = (
    glob.glob('/kaggle/input/**/fatformer_code.zip', recursive=True) +
    glob.glob('/kaggle/input/**/fatformer-code*.zip', recursive=True)
)
train_cands = glob.glob('/kaggle/input/**/train_crossdomain.py', recursive=True)

if zip_cands:
    print(f'[*] Giải nén mã nguồn từ gói zip: {zip_cands[0]}')
    with zipfile.ZipFile(zip_cands[0], 'r') as zf:
        zf.extractall(WORK_DIR)
elif train_cands:
    root_cand = os.path.abspath(os.path.join(os.path.dirname(train_cands[0]), '..', '..'))
    print(f'[*] Sao chép mã nguồn từ: {root_cand}')
    shutil.copytree(root_cand, WORK_DIR, dirs_exist_ok=True, ignore=ignore_heavy)
elif os.path.exists('/kaggle/input/fatformer-xla'):
    print('[*] Sao chép mã nguồn từ /kaggle/input/fatformer-xla...')
    shutil.copytree('/kaggle/input/fatformer-xla', WORK_DIR, dirs_exist_ok=True, ignore=ignore_heavy)
else:
    print('[*] Khởi tạo thư mục mã nguồn.')

# 2. Tự động bung công cụ eval_locked_benchmark.py mới nhất offline
os.makedirs(os.path.join(WORK_DIR, 'tools'), exist_ok=True)
eval_tool_path = os.path.join(WORK_DIR, 'tools', 'eval_locked_benchmark.py')
b64_eval_data = "eNrVPV1vG0ly7/oVHS58nrFJiqQt20scD5Al2dZZlrySvLhAIQaj4ZCcEzkzOzOUrdMJCC4/IBfcPeQhAZI7HJIAecjDPXmB5GGDPOY/KL8kVdXfM0NS9u3lw4DFYU91dXV1dXV1VXXzC9Z60GJBMoriSZ8tinHrGZZsNBqNjRd+8SLJ5mHW+snBdp+9jsI525ku/Ji9niY+O0nwWzxhO4skYD9gu0kEBVG4YK/8BTucTBHeOfXzC/ak/cTdaLFDP2VFCP+zBKrlCcsWBZskcZ+N/WJMTXlBluT5KJn7Uey9D6PJtMi9JJ5dtdNiCih2sea5D38CRDb154iHWg2INNHe4z47ZQPWbXeedbtNdvoAvnTaj552OojEj6dsEvmsB5WwapFAzVEUxv0NBv+6bdGRbp91n7GCyM4LtjMLAc55xl5uH+bsIet22G40Hi/yKIldqtmTNXt99oSdA0agMWS74STzR+GIOT9Nw4n3zaNOk/GnLfX0FJ7OZ4vMy7vyoddko+R97C1S5N6rBBnHvlmEBRuF8HnFRj4QB7xw3ixmRdTamfl53mdplkz8uMny4moWWk+AMLgK6NHAGPh52IriPIzzqIguQ3bpM38W+Tlz3h7uvmmyr7/yVD+b7PIbb6R73WKnEXCTODgHcjL4zHFYIxiaUTgrfOD89s6Op3jQoq/ES6v2BQgV9KWIYqDoin25dY89T5IiB5Qpc54PYCw7HRdGDCkOcORw6IOkYLMk0r1ptXCsvNFiPr8CIMB0gYJYTBcAN4mg4iJg54nAI0UA6v9k4RdctGQ/LoDT3yx8tnPyNfLkxydHhwCA1RKaIBvRPE2ygiW5fMqv1GMRzUP5/NM8ieXzZJacy2doBERdfvOzSepnebgxzhIg+CoNcyZencDnLDz052Ge+oGGgFkrQXajoGiygyiHv6cLgG6y7fiqyY7SAgbKnylq48U8vWIwunGqaE2yYGp9accxgcTl0vZ4EQccIwK8EKTQ20URzfL2yIcxl0TBcx4CRfhwkMD4Z0aFywhlSMIiy3PUAjkHebt/IF/tz/1JuLHx9vjox3s7p97x0RFO7iRvp34xbfvnOX468vtPkyhWX0ZRFgPbHM8bR7PQ89wma7TbDdfdiMbMQhiDJIHswRBSTa4K5Lc2zo+scGCSmpXcDUQz9XO/KDIHgfNilCygw40sDJJ4HE0WWdhwObIiu+IPEjMHbhugThhzZTxokDJucL0SfgjCtGB79AE802hSmPI1NIRZdncaAPgPp2EDlKJ38u75yd7pCQzOGb1tcF3UAFqkDjKfe/jlPJpQMa8g9RMHgynBHyf+QjyNwjAd+xdhY2O4sbG7/+LFu5P9o5qWJ4sItA3WmI3mXq/TMR69YDyRX7v4RlSZQRVvq+P1nlKb9BXew3/7O77nVUb+bBbiyzQezfHz8hulHInC7YMDr/vMIM9k00NW6QCvsbv38nh7d2/X+3r7eH/70OyXXEGwMbmG6Oen9CzWEf1IjBZrCZEl8EJDJzvH+29PgQJs47rURp81DmEN+/HbvZcsh1X9q8GjDnO2syIa+0HBnn145kpOKFrMOkWGBsI56vivBlsdG/apDRvDQvnV4KkCkp0AmJf+Is8j0MjPoYw5eTSZ+7geuDZsbylsz4CVbADYXXhsvUvZ8yhYnMPS4PR6j1nrR6zb7eEHfHMbGzcbG3xsvO2D/e2TPYNPQrr77MwQ9LdZAmNMMpElHhYORctqCmAFcz6c4LOoROW11XpWvZ5ZsWfV7JlzzOspRGKmIRY56VjjeTQRLUOZ1a6ailjBnJc7+CwqUXmJXD5rObFqBp/Ao+qin1lVxOzGGnqiw0AKeCizwJUKwArqCwDuwvML8YzlnnzxAhZNsGXRwgly3SxXEdSs0hYv1RMv08YOlm3vvsEPHya7xCLVC6IxVM3B7htD6+hCfCpVJXVkVJfqSaAwtBV80+/kKxNb1yCkqwnpKkJ0YdcgxNR8xA5bE76sUYxbHfhi1xeqUiPQuvNlnS6Fr91ODQ6LCKlvTRwGGfDVoIOrYxILqZh3QZ/ukUBAiRfKp9CYF6S7aRILJY5mr1TqxvhLeFPHYzXzO9QyzWW+JHil96XXrdLrln4/RPWzMQrHbBzFI7BhMrBtww9g5cH67ARgP0Zgb4Vg9aPldyaNvTOwmodDF5WYVcTXbDCyWMDIzlb11WIOtgRaQoEuwX9gHECLi1AVBh6YXTVGWOCamORbojh3qJJrY87CYpHFHJ96MfeLYBoifjSX2/jHCZoAGyyyHLYog9NsEVotiRq1uMs0CtizzpCjEGCHSRwKZs+SALji5YtzsF5h9DLnHOxYL4P9SB92U2BbiVcX4RUVLGM1bhPw8zSa633IfIG7EF9vLGGPmMBmDlbJ3A+mYvMFO44p39LAzghW0XQKixpsKQuGu2/Ag4VOefPmtq12xWgq6hkMPRaUBka9NwbH4goWEFk0KGd+e5a8DzPHJVnyUZbsNbI9CQtH86jJzvSXoTvkCL/AnTa5FcD2R1shgF0T+AcS2tx3kQP40GMOcmkTdsNyXnhUwEcvms/DUYTDpcUZaVR9GlYN39o6D6WNJf9ZGwqFDsjg3R5htwFmBiy0RMS1sBjTIMoRbjlaXXF4J5NbTWbZCaSorm+6osUGWT8qYAzszqi6pelKVRagSVNj8lNPVI0m4XMr1SqM4HhcBhWpihIqIEUIW7VxQzR5fQWxnFs2x75AN81XhhuF3A/gEpBzU42JYrAYINC+TeYJTr33Zxe1E0fJBsL3y7IwulMvS5qLGCylZKNGaQXo+mG76PR47gcX51DqwGb9TTJazOQQgk6Q78j7NYO5hV6Qkn+EaxsYTfby7btNWaMtNQqqR8+L4qjwPCcPZ+MmehQ8IiDMBz2DEfkixY62FbiWCazYhkXlEqQI6NyBp97IedRk3SdN9gg9Vhks7gNwWKX+iPai3VLlccCrHkRx6GdO9wl7QHY7/bVoEgyjRTTJ3vvZSND9waB1CthewC54tnAUbc4HY0YiwLR9GYXvnWk7j34WOh3wJbQMssSICOqcqavGhfwXuEALf4gjPvXAiALw9wXmQoC+sSmqxRCVfiCHbdVg4ESS5gD6hM5ouQJX0HAIL3PuGOuDuCcz7LI/y80pTsQTCnhJn6VXAgG+FY82gPLiAIT26MAAgycnDx1bwRoAxyHxFHdgTdp4uc1loKfJKax2CUjWUpBD+AtzCxDOwc04OOu0Hz/rPt56/ASkC563nj7rPd2ix86zXrfz9NGwWatorH+gGBFT78mzJ70vtx5j9d6T7qNOb+sZPT/devr0abc7NMgaGrLnebMwFsNUXWbhnaOZb1WDpRQL9QiPTMEl3T0wBu4M3g/V65l/Hs7EWMK2AL81hqbhZA2rrYg+4BiSZw8dlTFOTzU8WuuCAPVL43BVVWjRfALYaCK0kxQ6ywkipxyquMbQpTmHTrbG8cvnjer68UF2Uw20A1htuNVrgN2pn4UQa6jvlBiUD03OP2EVBiDEC1hXwZ/hZ1fePAQlFeTe+6iYwlo7i84dQoBqtIApyFvhsspFAnfD4ZJX4RyUpQ/NhhAOmSXgi1bhCw4wzcJ8msxG+rUIaJDtqac6vS2Zn690gCSeRqRnID4C9hUZlVAh4jZoPCGr8vRBxYpE+RTku2wAbVck+NpidWPuQyjH84MAtkidNvhNGwkMLu7BzLIs9M0CGwXu3i1of4HPW+JLKp5VpRs+a3IYC9i382EANomHTZPHBAj+GdrDvGjnybiY+x8cqyou+POB1PBEjKxBn2d9WK6G7SBdwBpHXnWxwAk+eTHaSOJLCUy0H44Qm2Pg/tFAj7Tb9nMMAzhRXIgqReohy3DpS9v5Yu44AgmMiAtBOMdsG8s4RXGpVhlKWBR8LJAi2c4mQ7bw6tBb18X1tYNMl+Qg7XXkdGvI6Spy7FplKM1wTQ7VkOTglxpylMzR5NiCl47q1EOFkKM3pBFbEJQo8jVJrmjVmAFYWtP8F+z4aKe1/W4HZ9b2W9FVWPewp0SzKIrDCRUhcWqbxgGBcNqjcRhrnoH0N5mPMsVngJJ9WwUnGTj0OW+BWlAwhSFfhorz4wsOBbOiuPJm0UXoUF0b5ozKhhIhxMdCYoWBU1eALnhYCXtMlUsjq1eehR59VQkUmWADDBx/aKEcbYLB3jHZgAO2gGJHQhO7ysPBV9A88Go40qpjCb6B2S9oZqYQnGlEug9FmnOswWKOHSnXV6JM4rwWuGPwETwdFBwjKlJUXg5+PEQ0GoyEwRBdWQfYV0eLq+YPcK1GfiuK3FLitK44qsSwckqanQMaZSaoofA5nCwwgQzFz4HU1DWA+GLA3yNbZ1Hq0BShyUHdci34tAqe1kHflJd7GYX2guh/ZYVvCrWhCOnD3pFqd+TLPAxHsvRxr2wSUDz4jJBK+6Din4KtxtoIfE0A3jYUgLE8pt1Gghz8IzV+7mMIm9Zjw5TQ2k++/yHbqpoWYmU/Rxkg587w/9dKjy4B8htw/WkMpWHJY5kHxjtXE4KPwO0owBqCP7A1hm3NwPiehekMwhsllyi421QfdIfOZBtDC7BW42lQUy1xg0VjX26vaC25xmSxKNA2S53dUgHVkiIW8mWGy4ZJz2qbpdRI16LHrloB1epe01NruWh65iWDRVorpO+UglZCQlOg7aewhxo5c27P2JMDacM1jhdwkoIIRP09ygIpQP7qDMbJAeS9LVpCtUwOVaUpZGDV1/ryaU2tdcsIaFDvyy3Qw+DMWgD9nCzYfIE5pUqwTSxyLW18vohmOF+DC5i28wRSixxw4gcXKbjGCtpCCs88anbz+yi8hBkkNTL/tsYTAuoMUtxgTwTJPnNMLwPleInfZeIaJF6J3DVWTlJrG5um6rY6zZB7jbMHQ6ZasLx2XL1CKHykHXNtTFgp68SSr89ywbWLxOEdlfoHE2ryLGgT41RaEecplXH7GZ88tJGAIaWcI0eLoGh00Ng52H/b/zo6bR1sdh839FqrhmCgnvRLi1RdvABHap7NSY3ZpRMfg12lFwDqQe6Mh4q724ENPL1xNzSTx8TlXVzRLpDVI0xFuKCUPIw0WNmNzDk5fsMetV6HWYz8echeUqOuZj3xBthi8MzR/BJ8hjEvCSU5t8thMBvELYuHQXlMPloJTis0yF+fXZdw3BgCUvG/QLy9CJXbA6byqExCE7RT6lHUCwzIQQOWMAhFmqI94POj7Gw/BXW/l2VJ9n20aAX0wJSNAU0MkkfYcI0OCh4saHDGU3mDkrbwqZ4E+jyzahiLnwC6di7OnvaH2O5FG9MTihy9Og5WAwc6yABtstiF22eXtJpfQCKkapk73hz3xpikOIWoUeo+b9nDLsjuoIs7KMp8nUeQvRJPvAC0YSGMJVheKLcrw+W+ISEgkpbDIJ0NjfVpEYcf0jBAy385AgOoDoeSwf/6m18Nec7uFB3R4J2DFOE3vPXBtUXoTZO9U1gH12UypHTaG1XV0J8M2Q62ALmVfVSLZIOi5vOvTOEvwEtVkXtMuAUwSztPaZL7EU5da+7y8TCUo3rZDi/9mWOFg7la5IsPvl3QAHI/vccHXNfvMxVt4ZpolRte+HZrliW+kGN42kMzT9n5T8QWAFTn+yS7CCGuJF4J+79+Oft+XIt891B2MDbtLU7pW50T8gRCbTN0QRaQiANsKygAPk1SUAULdJlgFBjSj43sXlhndW53bWSb87m6YZAEkXODQjXl72rbK8ZUuqfNOA1h18bCQD64Yhvoc8eCzm51RjLjVY/iQD/CrJ8uxuNZyKd90xzRgfEs5BbNS7W5kXsey14/E9Z50UFtCynHbfwjBBlVmEzchUw0SL92SmFKcHBfoRbjPal6/z+UJ4v8Z6CGfUjbXxSgyfNCwLZxF9BkYeyfwyZrYJaimdwIFiMfUnCrznlY0eU0heCbrc8VL6Txi6mzfK/FzVO3WkFuyUSNK8HXEGJoOaSiWywDX1PB7WxYkzTbeSehyNEU8P0i7F4wfqLHiJYIW8wUPmOLZSEU5WWMErwOJdc6PPgA+O4YlbA71jSJ0laVoSoGxrMBIFXFQD0ZhpfUnJyE5tL25AAI7Qp78Vjm2kQ4BXNHZ94IKx52T3LLq1Vfye1B2qpO49p6iEL/MuZPIVZaOGT0nzctD5lAFHa0uDJyBUA9Bj7pKDp6IWpx3aQOVOhzJKDZyNzEQAssnxz5Zof2pZsP2mk8kblEm13aHPJCdZpE1gdNOANN6czppAfZz65CR189zHOvx2y8rzRS0qea8UtShTRANYgptRH59DC0Z+gtckIYZaXcCg/WDHppJFfUNpWicYYoUisHKRUVidQYDHvK9KLMjHaezqIC027yMHUNdwZoc+Fe8OMrhzA4Dc7AhohN4WenYTRBzbsmCuERMFB0Vd6p/OzWoFA4vmDvFpj4FAtDi2JyUhonFLIDoQOxWtAuBmV1hKW4LvIzLLbdzDtFSTWcumrCD9GRheCihVnocIL6dQk7q5hSn5ujOUJLXC3MOWC5qLwJZ7rBWhYubVCM4roGyzmOjqyKKVCcbHdJ2qO1ao5jJJLEtZLeM46lWLZhxeG7CKeBsw070f5pKj9D/vA+PE9rl8JxJcGKT5RxXJtZJbqyLF9Kzke5El4bkXew+lJMDqbcAAi13rj1Q1MrTYpcNb3v0ET3Ru+XDcXOfgTBL5RbCjBpol3Yj6uQEy9xAdZcErQmiic89oTO02P6MNzQlMjjz8ZoZBjtbm4ac4gXjqRQAcI2LzIogvUIBqREZZNQG1aIxCTmg4FJ90RjMnpnLXdgl1ByiEGWazQCyz+s/GTMWJQ/tJo3+dMWNqgjq1Z8S/KF5c8zlPpDY7jFGo4HNqV1SafY0C6WJ9ra29lkMQ/j4i29cTCYlkUp3/nbjpgXEZ4sOyA3H3sOJ5Gmcz+7MM6Riq3kF7DvUzvDHzDpCDPab0PiGPpmqGGn0WrprSPMPLRFB8I/OPZhUR1ApvgppK6ebmq4fPNuB1MFTcvalT6w+mbJf9bqPgZEqm+i/2S+ZeQhQeSwfhegsRUzHq9s1rDfZMMiAiSblqdjp+EsHTQOdWLISJ2ylS2t7qGyBZc0JMJYsiGeZAIGIZ7LXdrWF/L0YL66l5AQj5qxnrkN2eipXE+TgHaT1YO9q7s4EsdXP6utmqPAq1sTU9+DAfRUy9wckm2DeOq2tzqd5QlsnCg4ro3b7AJtCB+2WJE0dEfqcPIL2MG19i4xYAIee9iRgDGCmEErz3zKtnXvyKRLP4MjWEVezygICzfWkXsfgO7zpEedD6m4GKNZRMb5OfVngUbTlRKcY/A5wZ4O9MLRooDN0SqaE4JAI3O1VoB9TL6693qbXztG3Scraxu7/9rqndW1dRymvnEM3a6dR+TbgPp+wFVzDjvP0IP9R6hke0c45VBR8ExQO314nYCgI6Ce0eQUQItAbJDhK+Yl+pd+NEM/giN8sMJVzI8k8FCFaI0+yB2vM6wo3DJoQKjqyydmdKDxYvv0xdHxm71jfrvBi/3D7QN2cLTzem+XPd873Hn1Zvv4NUjQ26OT09arox3Wa+3uv9k7xMOaAPn13vH+i/2dbTw5CQp5++S1uTpJ1+Zp9YICmHzoxYf7CbR+prsKrpHwtumsaz8e37Cfs1OpXfk1BgJO+ewQqtIsqvLzqC+AOdsR1w4fOZn+7VAUye2rxqUQ2Bg1Aw3XpXJl8K+O0ZIr86F2KXIWmV5cPNQBC6KwLPJkdole4osUt/B1Z5x0xjA1oDEZqR1/wMq9Bsv88aZZnUcRzpFN88fl+psX/mQyCzejGBTK5oMHn0aCOoskWQJMuiNLqjE2y6gwiiHSDSoCTbvNJRCSCSshhyW/ek1s1hrapt0tGZM1HKsl8VMCNPi+/kmj8dX+3ju812N3+/AVe7m/jXbA6fZbdrp3csp2Dva2D/8oLYt59GdxA4znWn0ElN1+++dE27//8gj+3378ux2i7vbjP5v0GdrJEReRqFPk7qppG6CV4+FB/2yUe+dXwqBAlXIjgwgcRqb4l0ZFeoR08YojZTaQ3u/ZjayOjI9gUUHNiYZL9SKWFE9GXC70AgS6rcDokR0vxx079BS37PZh/NJWv547eG5tyMonw/BfaX/boP6Ivl1DrRvvOrqRG3+5843YPda7qW720QOh84Med+wt+HBV9EyGiuliGIybVTl1TkFvit9d2+Ny81m8Qg81eggHNQclbfzkgqych5P1zdg4PxIm3rh1J6LI213vK8Y61qZ5UOJg9ZSCgfNs+CmSgJWMbKKkgG0xrzLGDBakETKDcMt+yZ1+FCdegrSNgUXog2uOAWARr4m6CyvgvAyRCEHLXP1L9JB0hpW48veAFxOvhkvCxacyblvQFg//XFdY1G/eCL87yOU1+T7MPrs3m4YIg8hab2/sNIFSv0pn/qwIc6CnBe3KVIwZaIFWSphuGq7lMKwZaCs729RhqwLaWB77SSnogFRhKBtCDZBTQKEH08Yuqzp71tbqbkyXqfh97zC3P0sXft/68A46ketFsVir0DK/VsrsQw6bC2OS82JMTC4HVvWbanj1DrxDLSQPzi2bS+JQNCVclJ3Rqv46D/Sdo2tAyKq8BYUP7bUB/bU35QQ7UHTZL7kFNzDzFnR+mIp802JghL8tODP2TYBGQbMs3Cr+bkh56fifEbssb6VKkCqIaW+lFJAx88vyIr3bmvluDWwp7myMjoF6AhmPKUXlYKQaaMo1xOKIcmbeE8T3vfpWCDPNCJbZ9+pGGmXGE26YX7qN0tkq4UyiEGLpFQ6DWEbhPapnk/7m8mNeQjLPjMLSuUoz/V9Bq7Lh8vNfCliVlYH5cQAFh18rIKkFkVYAxMTx8hBx8YRUUYYJqcZBs7KmJC0jh1uMSjWlimGA+uxaD0r/y/xmyFA79rtb+Q1zrsvs7v9oi6+WLuze3yBf8fo8XK5EP+4rZt8f9tu98c09AGyUVOW4cYzBIaOWZLpRCW/OMUEkq1fihdNOZh1guwG+/dZ6l6pXzrVgK33P3YZx4BWc3oGHx3Yh4PUetBnnFi0Y6toN+uZWLi3BCrU3gBzK0MDy84r2jOHNLZ8yioiVcwcNwezMKhzymDeta0DtqumkzqoQM84ya17ZaIbl49LVczYmHj3j1qGpnMQx0ei5uA5N6ayOwkCzdG3ltL5uur5q3YQWg2K+qgyKPdnpETzONKvJLMh0BdvOAEEE3FyWhpTrRHqdWxh4fcnnoDCvA+IWAIDzSkgVGhxqzigym6Jp+HwDXgMPvpzIVGZdH0my6isam2a7EolyNFQxifNeFjKrY4DjiMModEfgRQV7qmE5KOTBsSVZWOUF2cycqhh3SzOyKi3V5WdV1vP6ttbmanFQOk8TRGbGVuVg2SouNJdTXZ/AtdwIWmYAGUn6mjhhm+kCI+GLUwRnNshGV5av6CrZu5XDH03m0Jk7OnFXN/D8OExVtGz9h3u/qvTRGKjMTtvr1jK9bnZ1nDX1rhXuckNp5bf+8hVazTLYqz5zea2+Xp7RHithr1+n8axwHWx5Ga0HMtfTyltj0QEwYUM06jqPU35t55UuEBxQegJ36zAjlvceIe/aew67pvcCaEnv+du7915IzloGCF0lTTRTs5HDwl0pAlKEa/jg4DnLnX24SO7ank5wPxkHabLym654M3SXGWXL2l/DWw23hL0KYBWHxbxrWe7nP5Yzv2c485+w5/t7h+z01R6TV6gy5+vtU3bwp+4fhQidFlB2NICZMcd8XNj9i2i4drSDsl1AwAR35yq8j2EUSN2XYTQj26DuCFEVClKi8Co222OvvO51LVZ873XRiZoIRc+KUCDLbz/+KzL99ttf8DiF5j0AffvX7DnUO2RfQyADRuK7v8XL4vcOTrfZ7vHRW7dRacgOW4ge2D2WCQZkH6FJYZv98rVXJNRnvIO87lrdFbfY1KA4u2zjoZ3UMfy69WSJlNNGs0G2gqpnJ59CtTuFWmxpWBFusQXCTloE0vA0b5MeyOkA1Icwi9E6CJ1yd90KNzzM2gJK664OppVeItZNuLXe2D+Lz64FNXiJ/s0madQKAbAx/vdf0um96Pbbv5ijz/b24z/hFYi3H/8N00CgtM/uX8vG2ouUrhu7uQ8aS9J7I07wVTojnHN6Ttq5lGpoqs5RrvRMR/qSW671nbL8MtJhbdhHuHtsrPU5lgbdq12xy92x2C/FsZvP8svW+GZ7Nb5ZHbdaHYP5QsQMxNV7MsNnKlPpcdzlbZd0Jaa/UccXHo6y8mWrU2KZZBouBDscJTAvyTzWt5QuH4ZU51vrq0zXktl48KBhELv85tO1dyymdxmZdfeo1jNbX6O68X1wUzvJMH7yevrd7zE+8t0/o3DcfvwdhJSm//EvbH777e8COiRt6gFDDcD0L24//j0cFRTNwRFBWH/+En/Jod2o50bVyf4ZQZN1gVFBTk1E9A+MjK6PkK68206GT1cCmaFVEtgVGYqrMWEy9uBxbynMStnGLvZXoq/RkXbAVgJht6UvZlgXKBNXKeLQG0hlPLRKxZwOsfwBgZi1ARk7MFMTlLlDcOZTgzSfHqypC9qo5bQe9O6hmzuHcJaL0/cQdlEzrib0onyPwqCBxVQZRisByV4R0Pi4BHpdZOcOEZ7PivTUR3xWxHpqYz7Loz21UZ/l8Z5y3Kc24lOK/NTFfD459lONAdVpFRUKMsNAenram0YLeKNsHEG+wDzkV3+TaVS2Vs1mq1ppUudwtvTfSpe1RfZnoCq7rm3SqF+DuwY6eFfAyY/tiH4pf1+V2E9DPjKRj9YgTz4RucmkoVsViGw5vrpQzTp04+Xo6kI269DBxFqOsBTBWYsrXYEqXYWpZhbBj4t5oyyhg3IcbavqSq5WrHpmzqSGHv6Pq3NSwXzTJ1Sw2XH3Top7dZjxLvjIn811Ltd8fG7SxUz1Ncj/a9YYrathXxLIKyXrKhlrB6+RrathLB+8xnhdDb6CcGAp66vAUw2drmWTElLNKFV0x3WllD0gxa//DHMHnOuKzFD6nMylAyeviLqJvAHOcuna/c9fQ5GmqP+Qyuu8ytq7jMsD1JqYiH5OAQLEZZeKtIPMLhWZBmOjdMUNMegwoh2h+BWMgl/wUjqTBbu8hH6tbyR/3ZH/tiP8eoQ8c+UqT9D374v+yTvwMr/YP9gD/xh7vn3EduD/6dHhS/bq6O0fpU3YHs6Bj/gjA9yHoE8ywc4DvYJeciHcBfrnNuhXDuUPHPLsYPh5QzOYl1/WOlJM7OBaglOaeIso5FsHcB5CHNP0wCWQRR/agEOMJ/3mzge8bIzHD3BnDJSV4idrI3JlNOr+O7vC+ujWHRBhhTsFiu6AS9QRAzAN8ZaVXHFCmkpNZaw3bdXeNK1t87bspqHnxL3Y/Ebspm3IDvU9MHTpuhpigHvfwBtaSr/0h7/paLBr3H6fwT4THdlcFKwu4LlsDBrY+eMZ/2ELmzelKwjKaHO8nOpsOuSO9SliWNGS4UimpONdn30wxZqyZPlpKy7hmd9XIbT88kZrgV5pQqgfKcWf/OQXLEhJKBvtfeOO4cmnzBpxTEbhq8wZU1RGdFO07V5UpkizZGw02VqBWpZt9DnCtVF1JmpJE0y5o5ytlDVAVZU0S9qWj81qiaOwBVxF14AbGKqSV9uune6+QvKUKJHwCXYYovdIiB566a2fluWp83AtljgPRpMIf0H2EwVMa2WOo404RE/wEUIDV3jvk2XyNiBbz0cfFZgrpQQ58Ftd4I8mypP4y47q/wBPSsLp2zjnvxK7x11flEZUwghQcBneHG0juogJBmVMlzE17v1p6968dW/E7r3q33vTvwepS6XKxmn+vuIL/hwP/dyrdfzM5a5D86whd+0cbm5XaNKeJ08ev6ffCFztl2ooD1RdrWp+Df89RpX/g3f4RugwzmUdI99m2aSFZBrICMBdg15RZYZAmVvSGUen8zC+Yx8X4Beg6HjkkogxZ5tdV+fJGacI+ftQj3xFmsQmrPJCCKFY0Clf11rim0ugzV/kKy3my6qIxdmuIAqtKje1iaBInGXG1HJCqaY7MMPYkfJdXalwKRll/acpWT2SKi1W5Dhq9W2onTuqcARtg0Skjqlc4L4a8KPHIzj/jT/mBEoBJ5afB1Fk3nxZr1WlQiTtKLUi16cGfYZOfdzGXzpBL1nEr5RQlQL6pfoAf6le/uL33Y5lPr/9+Bsw4k9vv/012vK33/7mLXsNeRC/eAPpJ+9uP/7jIXv96rtfbbMTgMAigNp5B0kPO8w6Yb7sjLjd6CenJq3PSvqsnKRqStH/9Sylhj5hWp04yzJdWvWZLuPG9f3nRnTxfv+H3c4Nu77/5rvfY3Dxt9DElYg/wrtH9I5nB95zFTBuiK0CuQ8XZT0sw/0xDo0Cwq2xVfCfv0bR+h1cNg3t/XbOa1ZzZipdqW6nKKhYzv0tlVVEzuYKmdOaHbff/hUcXctuP/4D3V2Gpg8yZSGZsjIHEpDgyK3JFayDqhVcYMtyQOO0w0qExpEHBXcfPMAdcJVQb90781/k3EC4l9I7KjJZH0hUrAYowWZgm2nlI2mPRLl025nkQrH0zZWKDQecya+8nj15PTcQvXIXwYuHpvdmGT8qeV34i/UeebE8j+IEnkd3I3gij4vfJrXx3/dxehI="
with open(eval_tool_path, 'w', encoding='utf-8') as f:
    f.write(zlib.decompress(base64.b64decode(b64_eval_data.encode('ascii'))).decode('utf-8'))
print(f'[✓] Đã tạo script kiểm chuẩn: {eval_tool_path} ({os.path.getsize(eval_tool_path):,} bytes)')

# 3. Tự động bung công cụ plot_three_way_benchmark.py mới nhất offline
plot_tool_path = os.path.join(WORK_DIR, 'tools', 'plot_three_way_benchmark.py')
b64_plot_data = "eNqNWMtu48gV3esrLtgwQE5LHJJ6tCRAC1vuTidpP2JrJgEcgSiRJZExVUWQRbc0g1n1YpbJYJBFkCwyCbJIgCyCLALYiyw8P6J8SW4VH6JkudG0JJJV91W3bp1T5RfQ+qwFHvdDthhCJuatvmxpaJrWeEPEG54sadL61bvjIXxJ4SSkGZxy/IQwDuRLG04oA30SJJS2fknWMObLmCREhHdUdnnBkiS3BkqQ9BZ6Zs9oXHNICQtAJJkHfkjZsGGbcEJSGoWMwk+4B+MvL6/AsZwO6Ge20XBMOOMQhKj0BkVEhmKXEWFwXNjtmjZ8DmeO0WhvRU8JnKF1mHCUPJVP+lkY0VRwVO9J+Q68IVE0I96toQbcCJcxTwTwtHxK19Ujy5bxGkgKLC6blkTEERdRODPjtXyS3XEkGo3Lq4ufvR5P3KuLiwmM0KAZExGYZJbKu16+/4aHrHrxw4SRJdVdd45huq7RBM00NcNoXHwxccdvj68m7uXx5G3NntKv+0KV0+PJ8fXriYaPEV+k8i4wRy7m3hVymtz3ZO16aprClDMzZgvNwLHPISApESLRcdRmKnyeCVROqMfZPFxkCdWMYQPw2vabtV4dZ1vV0UhTdSSNNnw6xzRhlIUqZgdV1xE1s5TqWkrJjCesdWe5/db7IBR0kYS+BhjM832sZoXckTAis4gCjVIKGvojWSTQt/SGgTVBJyu7CWTlGJg5pZnN5GSlOjY7TSmUhl/RkW73mlihXcy7H4ejtmXhAKSVF4D1eZ0rgT3EUk2w+oksj6KUu+AFoXzzUCLioVLzCEbMk5Cm6PdGNclLG0cUC/KMeAmH4/H410yfbB5+zxYggs3DB0Nr7ou+Ibe0kDwPNvf/ZLhqNg/fyqKez7M05OyA0hUlUaF0srn/KwfBH//MYLl5+C0wyg9pXIxbx19IhSs5pgX6+GMIcfD4Fwbi8Qcv2NH58Xebhz/ABCdjjCGt4fX4NWqO0YcayP3fYvDyl83Dd6JUneYZXdpu6vEkz0zXMi2rCfmvbeX3rm32cTo6fdN+NVVzgA4g9UhEfVihFPgUEGK8IFMOCf76HI5y807NfH9gOmip75gOTvagZ7bR/KBr9nH2u2a3N81VOjWVQS6Lmm0UGnTMQU9p9rtYMOZgUIxihbIsNhHt2ILqEWX6dsqNvADfh74IUMwynW6uhAtGpDY2YVWaM5LoK2jlYs1tWpplC5Y2jUbaFg8rmJTAiGvb4xFPRtqLvif/sIH6C1o2dtVVrAXl2Kk7bm4Tte+wQNcKbfdgtua4bfedmb/n2Op37YFXd9zeGfHLasSdZwLY4R6J1PrL0zAVYRTByzFnaT0E2uvZlr0XwgzXr2Vp1RL+KcOKJkg6IX6pXEJY47hcc5zgSR6mBJebfIqaRcaKe3s6rIq/FJfSqnPbJS854bLZXFDhBjRcBEI3diRkJghjXGC96HPt62Bo2vNvjmrr68m1Wo/0yuhKNzCH1atKHTZ9LuEsMD5qRdCVGOm4BNofk5NSHueJn440Pp+nVECMXCPSj4UYkJHmUSZognNxhy8zLgRf4suco6rE2FdmN397r9IiRSK/nCSZFfTkilBEyA0SYb5T+4zN/X8zmCSbh395iHkK+nDf8fgPNtyuh7u03BLgE9bLz4PHfxO4RmSt+7edA+6bEBMfu4ydKFYi9G5TfXWgVdVoWlvtNQ+DZ0ZYN7JWBnQNcf8/8A7HA/qRsROm/Qk2wqWcRtvubjsiuqDM1yPuIQfHMU0gonNRtzzA5wR3GSSKcbosc1AtEGfLcc4QrohfsdxMbvTkRg1mGW7OWIDbPCYBN8q8wrWDG4Elv6N6GYsEGiRWk/i+W/BtQbf4iXlEktEkyWjhPZHeXMznx6myzj8V91X0+CyjPe04vqy3Kaq8RBCiyR1uXDnbUZDbGx+3NgiCdgtJaEtl8vccA5bIvx1BkQG2iHIyYbgs5xEnQj834DNw8IukEYcKRZiCEMUf58a0rvlyVDzdDO1pOUfnCIkkwnmEJUUk81JkdbBaSId5Gt2l/YRRCybtOorxSn7VbaQdy+z07VcyKmyeliacA6yZ0yVSoNOvOLQwYXW7vX0TnQMsmtOnvDkVp5YmnMGgMrEdCaZA3lUCqtjyRqfe2CkaO9tUyYpUNZfnsKkMIbkgUiiwHNlymcpXtY0cacNDXPqEfRXpGs96cHY99Hc8tFqHWHOXb3XJrc/b79TtO/v2DzFiYX4HD0GvTj01X3jmiPZ8PbFWgoaEykpRIXZABXFzotDz6kYuMg7I4CEHWQvXmN6yd/sLvC2LvmVPD/QXyLtdbZ+MvLmRde7kxsH67eC3h99+vu2cHpAs3N1ojnUkD1Kd/NbLb/38hrpH2nSbrp66sEdF1d8zW8K21d1LT056J0hvHzKQ3Pd9AcInhMMvsscfBJz/+AFR953iwWuSwVvJhRmidLa5/zv7VJ6rRfSULxIpj5KzGV+5grsET/A4Kt022xi0aRufwCXyjCWkHTciazwkFqSAh9YlQjVWQPrkzLt7wkUndIW7PZffFixRHR3JHUVa2ZOvDmxF3CEGTXHnooLQttpexNOSouIEtzO497r535++n8r/FawyUlIdx08oj3TqHxwzKiF6CF/vOv1GnW/xoOq6cgiuC6MRaK4rT7uuq+Vbwvzo2/g/FeciVw=="
with open(plot_tool_path, 'w', encoding='utf-8') as f:
    f.write(zlib.decompress(base64.b64decode(b64_plot_data.encode('ascii'))).decode('utf-8'))
print(f'[✓] Đã tạo script vẽ biểu đồ: {plot_tool_path} ({os.path.getsize(plot_tool_path):,} bytes)')

# 4. Xử lý vocab BPE tokenizer offline nếu có
clip_dir = os.path.join(WORK_DIR, 'src', 'models', 'clip')
os.makedirs(clip_dir, exist_ok=True)
target_bpe = os.path.join(clip_dir, 'bpe_simple_vocab_16e6.txt.gz')
bpe_cands = glob.glob('/kaggle/**/bpe_simple_vocab_16e6.txt.gz', recursive=True)
if bpe_cands and not os.path.exists(target_bpe):
    shutil.copy2(bpe_cands[0], target_bpe)
    print(f'[✓] Đã sao chép BPE vocab sang: {target_bpe}')

if WORK_DIR not in sys.path:
    sys.path.insert(0, WORK_DIR)
print('[✓] Hoàn tất Bước 1: Mã nguồn và công cụ đã sẵn sàng!')



## 🔍 BƯỚC 2: TỰ ĐỘNG DÒ TÌM CHECKPOINT M4, CLIP BACKBONE & TẬP TEST
> Quét toàn bộ `/kaggle/input` để định vị checkpoint rút gọn `fatformer_crossdomain_weights_only.pth`, `ViT-L-14.pt`, và dữ liệu kiểm thử (Clean + Degraded).

In [ ]:
# ==============================================================================
# BƯỚC 2: DÒ TÌM CHECKPOINT VÀ TẬP TEST (CLEAN & DEGRADED)
# ==============================================================================
def find_file(patterns):
    for pat in patterns:
        matches = glob.glob(pat, recursive=True)
        if matches:
            return os.path.abspath(matches[0])
    return None

def find_dir(patterns):
    for pat in patterns:
        matches = glob.glob(pat, recursive=True)
        for m in matches:
            if os.path.isdir(m) and len(os.listdir(m)) > 0:
                return os.path.abspath(m)
    return None

# Dò tìm checkpoint M4
ckpt_path = find_file([
    '/kaggle/input/**/fatformer_crossdomain_weights_only.pth',
    '/kaggle/input/**/crossdomain_model_best_m4.pth',
    '/kaggle/input/**/fatformer_plan_a_best_weights.pth',
    f'{WORK_DIR}/DATASET/checkpoints/fatformer_crossdomain_weights_only.pth',
    f'{WORK_DIR}/DATASET/checkpoints/m4/crossdomain_model_best_m4.pth'
])
print(f'[1] Checkpoint Khóa Sổ: {ckpt_path}')

# Dò tìm CLIP
clip_path = find_file([
    '/kaggle/input/**/ViT-L-14.pt',
    f'{WORK_DIR}/DATASET/pretrained/ViT-L-14.pt',
    f'{WORK_DIR}/pretrained/ViT-L-14.pt',
    '/kaggle/working/**/ViT-L-14.pt'
])
print(f'[2] CLIP ViT-L/14 Backbone: {clip_path}')

# Dò tìm thư mục test benchmark clean hoặc giải nén nếu có file tar
test_root = find_dir([
    '/kaggle/input/**/test_benchmark_gans',
    '/kaggle/input/**/test_benchmark',
    '/kaggle/input/**/CNN_synth_testset',
    '/kaggle/input/**/test_clean',
    f'{WORK_DIR}/DATASET/test_benchmark'
])

if not test_root:
    tar_cands = glob.glob('/kaggle/input/**/test_benchmark*.tar', recursive=True) + \
                glob.glob('/kaggle/input/**/test_clean*.tar', recursive=True)
    if tar_cands:
        import tarfile
        ext_target = '/kaggle/working/test_benchmark'
        os.makedirs(ext_target, exist_ok=True)
        print(f'[*] Tự động giải nén dữ liệu kiểm thử Clean từ: {tar_cands[0]}...')
        with tarfile.open(tar_cands[0], 'r') as tf:
            tf.extractall(ext_target)
        test_root = ext_target

print(f'[3] Thư mục Dữ Liệu Clean: {test_root or "[Không tìm thấy -> Chế độ Dummy]" }')

# Dò tìm thư mục test degraded
deg_root = find_dir([
    '/kaggle/input/**/test_degraded',
    f'{WORK_DIR}/DATASET/test_degraded'
])

if not deg_root:
    tar_deg_cands = glob.glob('/kaggle/input/**/test_degraded*.tar', recursive=True)
    if tar_deg_cands:
        import tarfile
        ext_target_deg = '/kaggle/working/test_degraded'
        os.makedirs(ext_target_deg, exist_ok=True)
        print(f'[*] Tự động giải nén dữ liệu Degraded từ: {tar_deg_cands[0]}...')
        with tarfile.open(tar_deg_cands[0], 'r') as tf:
            tf.extractall(ext_target_deg)
        deg_root = ext_target_deg

print(f'[4] Thư mục Dữ Liệu Degraded: {deg_root or "[Không tìm thấy]" }')



## ⚡ BƯỚC 3: THỰC THI KIỂM CHUẨN KHÓA SỔ 2 CHIỀU & TÍNH KHOẢNG TIN CẬY 95% BOOTSTRAP
> Áp dụng cố định $T=1.0811, T^*=0.3700$, kiểm chuẩn trên Clean 18 subsets và Degraded 6 variants, tính khoảng tin cậy 95% Bootstrap.

In [ ]:
# ==============================================================================
# BƯỚC 3: CHẠY KIỂM CHUẨN KHÓA SỔ 2 CHIỀU (LOCKED BENCHMARK)
# ==============================================================================
import subprocess
import torch

OUTPUT_LOGS = '/kaggle/working/output_logs_task_6.6'
os.makedirs(OUTPUT_LOGS, exist_ok=True)

device_mode = 'cuda' if torch.cuda.is_available() else 'cpu'
print(f'[*] Thiết bị tính toán: {device_mode}')

cmd = [
    sys.executable, '-u', f'{WORK_DIR}/tools/eval_locked_benchmark.py',
    '--temperature', '1.0811',
    '--threshold', '0.3700',
    '--output_dir', OUTPUT_LOGS,
    '--batch_size', '16',
    '--num_workers', '0',
    '--n_bootstrap', '1000',
    '--device', device_mode
]

if ckpt_path:
    cmd.extend(['--checkpoint', ckpt_path])
if clip_path:
    cmd.extend(['--clip_path', clip_path])

has_clean = test_root and os.path.exists(test_root)
has_deg = deg_root and os.path.exists(deg_root)

if has_clean:
    cmd.extend(['--test_root', test_root])
if has_deg:
    cmd.extend(['--degraded_root', deg_root, '--samples_per_degraded_subset', '500'])

if not has_clean and not has_deg:
    print('[!] CẢNH BÁO: Không có dữ liệu test -> Chạy chế độ Dummy')
    cmd.append('--test_dummy')

print('=' * 80)
print(f'[*] Lệnh thực thi: {" ".join(cmd)}')
print('=' * 80)

process = subprocess.Popen(cmd, stdout=subprocess.PIPE, stderr=subprocess.STDOUT, text=True)
for line in process.stdout:
    print(line, end='')
process.wait()

if process.returncode != 0:
    raise RuntimeError(f'Tiến trình kết thúc với mã lỗi: {process.returncode}')
else:
    print('\n[✓] HOÀN TẤT VÒNG ĐO ĐẠC KIỂM CHUẨN CHUNG CUỘC 2 CHIỀU!')



## 📊 BƯỚC 4: TỔNG HỢP KẾT QUẢ, VẼ BIỂU ĐỒ ĐỐI CHIẾU 3 BÊN & XUẤT ARCHIVE TẢI VỀ
> Đọc bảng CSV kết quả Clean và Degraded, vẽ biểu đồ đối chiếu 3 bên (CVPR 2024 vs Plan A vs M4), và nén toàn bộ thành `task_6.6_final_results.zip` ngay tại `/kaggle/working` để tải về.

In [ ]:
# ==============================================================================
# BƯỚC 4: BẢNG TỔNG HỢP ĐỐI CHIẾU VÀ TẠO TỆP ZIP TẢI VỀ
# ==============================================================================
import pandas as pd
from IPython.display import display, Markdown

res_clean_csv = os.path.join(OUTPUT_LOGS, 'final_posthoc_benchmark_matrix.csv')
res_deg_csv = os.path.join(OUTPUT_LOGS, 'final_locked_degraded_matrix.csv')
res_json = os.path.join(OUTPUT_LOGS, 'final_locked_benchmark_report.json')

if os.path.exists(res_clean_csv):
    df_clean = pd.read_csv(res_clean_csv)
    display(Markdown('### 📊 1. KẾT QUẢ KIỂM CHUẨN KHÓA SỔ TRÊN 18 TẬP TEST CLEAN:'))
    display(df_clean)

if os.path.exists(res_deg_csv):
    df_deg = pd.read_csv(res_deg_csv)
    display(Markdown('### 📉 2. KẾT QUẢ KIỂM CHUẨN KHÓA SỔ TRÊN 6 BIẾN THỂ DEGRADED:'))
    display(df_deg)

# Vẽ biểu đồ 3 bên nếu có script
plot_script = f'{WORK_DIR}/tools/plot_three_way_benchmark.py'
if os.path.exists(plot_script):
    try:
        subprocess.run([sys.executable, plot_script], check=True)
        chart_p = f'{WORK_DIR}/DATASET/logs/task_6.6_three_way_comparison.png'
        if os.path.exists(chart_p):
            shutil.copy2(chart_p, os.path.join(OUTPUT_LOGS, 'task_6.6_three_way_comparison.png'))
            shutil.copy2(chart_p, '/kaggle/working/task_6.6_three_way_comparison.png')
            print('[✓] Đã tạo biểu đồ đối chiếu 3 bên: task_6.6_three_way_comparison.png')
    except Exception as e:
        print(f'[!] Cảnh báo vẽ biểu đồ: {e}')

# Sao chép các tệp CSV và JSON trực tiếp ra /kaggle/working để Kaggle hiển thị trong tab Data/Output
for fn in os.listdir(OUTPUT_LOGS):
    src_fp = os.path.join(OUTPUT_LOGS, fn)
    dst_fp = os.path.join('/kaggle/working', fn)
    if os.path.isfile(src_fp):
        shutil.copy2(src_fp, dst_fp)

# Đóng gói toàn bộ kết quả vào tệp zip tải về
export_zip = '/kaggle/working/task_6.6_final_results.zip'
with zipfile.ZipFile(export_zip, 'w', zipfile.ZIP_DEFLATED) as zf:
    for root, _, files in os.walk(OUTPUT_LOGS):
        for fn in files:
            fp = os.path.join(root, fn)
            arcname = os.path.relpath(fp, OUTPUT_LOGS)
            zf.write(fp, arcname)

print(f'\n[✓] ĐÃ XUẤT TỆP ZIP TỔNG HỢP: {export_zip} ({os.path.getsize(export_zip):,} bytes)')
print('[✓] Các file kết quả CSV, PNG và ZIP hiện đã nằm trong thư mục /kaggle/working (tab Output).')
print('\n🎉 HOÀN TẤT TOÀN DIỆN VÒNG ĐO ĐẠC TASK 6.6!')

